# CL-EPIDTN Recommendation Engine - Improved 7

This notebook is based on `recommendation engine improved_5.ipynb`, but imports `cl_epidtn_recommender_improved_8.py`.

**Key change from improved_5**: The data pipeline expands the item catalog with TMDB-only movies and RAWG-only games so that items like GTA IV can be recommended even without interaction history. The model architecture, training loop, and all other code are identical to improved_5.

Catalog-only items are recommendable via:
- Content-based token overlap scoring (`content_boost`)
- Text embedding similarity (`text_weight`)
- Title token matching (`title_boost`)

## Colab Setup and Data Download

Run this section first in Google Colab. It downloads MovieLens, the Kaggle Steam dataset, and both Hugging Face enrichment datasets into the runtime. It also asks you to upload `cl_epidtn_recommender_improved_8.py` and `users_ratings.csv` if they are not already present.

The RAWG Hugging Face dataset is large, so this can take a while on a fresh runtime.

In [1]:
# Setup for local execution
from pathlib import Path
import os, shutil, sys, subprocess

DATA_DIR = Path.cwd() / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE = DATA_DIR / 'hf_cache'
HF_HOME  = DATA_DIR / 'hf_home'
HF_CACHE.mkdir(exist_ok=True)
HF_HOME.mkdir(exist_ok=True)

# Install required packages locally if missing
try:
    import datasets
    import sentence_transformers
    import tqdm
except ImportError:
    print('Installing missing packages...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'datasets', 'sentence-transformers', 'tqdm'])

print("Checking datasets...")

# -- MovieLens 32M -------------------------------------------------------
if not (DATA_DIR / 'ml-32m.zip').exists():
    print('Downloading MovieLens 32M ...')
    import urllib.request
    urllib.request.urlretrieve(
        'https://files.grouplens.org/datasets/movielens/ml-32m.zip',
        DATA_DIR / 'ml-32m.zip'
    )
    print('Done.')
else:
    print('MovieLens 32M already exists.')

# -- Steam dataset (Kaggle) ----------------------------------------------
if not (DATA_DIR / 'archive (11).zip').exists():
    print('Please download your Steam archive (11).zip from Kaggle manually and place it in the data folder.')
else:
    print('Steam dataset already exists.')

# -- TMDB HF dataset (ada-datadruids/full_tmdb_movies_dataset) -----------
try:
    from datasets import load_dataset
    print('Downloading TMDB dataset ...')
    load_dataset('ada-datadruids/full_tmdb_movies_dataset', split='train', cache_dir=str(HF_CACHE))
    print('TMDB done.')
except Exception as e:
    print(f"Error downloading TMDB: {e}")

# -- RAWG HF dataset (atalaydenknalbant/rawg-games-dataset) --------------
try:
    from datasets import load_dataset
    print('Downloading RAWG dataset ...')
    load_dataset('atalaydenknalbant/rawg-games-dataset', split='train', cache_dir=str(HF_CACHE))
    print('RAWG done.')
except Exception as e:
    print(f"Error downloading RAWG: {e}")

print('Setup complete.')


f:\Questro-ASP.NET-REACT\recommender\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checking datasets...
MovieLens 32M already exists.
Steam dataset already exists.


TMDB done.
RAWG done.
Setup complete.


In [ ]:
from pathlib import Path
import importlib
import pickle
import pandas as pd
import torch
from torch.utils.data import DataLoader

import cl_epidtn_recommender_improved_8 as rec
importlib.reload(rec)
from cl_epidtn_recommender_improved_8 import *

try:
    import google.colab  # noqa: F401
    ROOT = Path('/content/data')
except ImportError:
    ROOT = Path.cwd()

HF_CACHE = DATA_DIR / 'hf_cache'
HF_HOME  = DATA_DIR / 'hf_home'
ARTIFACTS = DATA_DIR / 'artifacts_improved8'
ARTIFACTS.mkdir(exist_ok=True)

# Download TMDB dataset if not already cached (idempotent)
download_tmdb_hf_dataset(HF_CACHE)

def first_existing_path(*paths):
    for p in paths:
        if Path(p).exists():
            return str(p)
    return str(paths[-1])


[tmdb] downloading ada-datadruids/full_tmdb_movies_dataset …
[tmdb] download complete.


## Configuration

The defaults below keep the first run practical. Increase row caps, epochs, and `max_train_samples` once the pipeline is verified.

In [3]:
cfg = RecConfig(
    movielens_zip=first_existing_path(DATA_DIR / "ml-32m.zip", "ml-32m.zip"),
    steam_zip=first_existing_path(DATA_DIR / "game-recommendations-on-steam.zip", DATA_DIR / "archive (11).zip", "archive (11).zip"),
    hf_cache_dir=str(HF_CACHE),
    hf_home_dir=str(HF_HOME),
    survey_csv=first_existing_path("users_ratings.csv", DATA_DIR / "users_ratings.csv"),
)
set_seed(42)
cfg

RecConfig(movielens_zip='f:\\Questro-ASP.NET-REACT\\machine-learning\\data\\ml-32m.zip', steam_zip='f:\\Questro-ASP.NET-REACT\\machine-learning\\data\\archive (11).zip', max_seq_len=30, min_movie_rating=3.5, min_steam_hours=1.0, max_movielens_rows=2000000, max_steam_rows=2000000, max_train_samples=1000000, min_user_events=4, min_item_interactions=5, embedding_dim=128, transformer_layers=5, attention_heads=4, dropout=0.15, batch_size=512, epochs=10, lr=0.002, temperature=0.07, contrastive_weight=0.15, amm_weight=0.01, gradient_clip_norm=1.0, use_causal_attention=False, hf_cache_dir='f:\\Questro-ASP.NET-REACT\\machine-learning\\data\\hf_cache', hf_home_dir='f:\\Questro-ASP.NET-REACT\\machine-learning\\data\\hf_home', survey_csv='f:\\Questro-ASP.NET-REACT\\machine-learning\\data\\users_ratings.csv', device='cuda')

## Load and Enrich Public Data

This uses MovieLens + Steam interactions, then enriches item metadata from the local Hugging Face Arrow caches. No TMDB or RAWG API key is needed for this path.

**improved_8 addition**: After enrichment, the data pipeline also adds TMDB-only movies and RAWG-only games that have no interaction history.

In [4]:
assert Path(cfg.movielens_zip).exists(), cfg.movielens_zip
assert Path(cfg.steam_zip).exists(), cfg.steam_zip

interactions, item_meta = build_public_pretraining_data(cfg, enrich_from_hf=True)
print("interactions:", interactions.shape)
print("item_meta:", item_meta.shape)
print("HF enrichment stats:", item_meta.attrs.get("hf_enrichment_stats"))
print("\nCatalog breakdown:")
print(item_meta["domain"].value_counts())
print("\nTMDB-only movies:", item_meta["item_key"].str.startswith("movie:tmdb_").sum())
print("RAWG-only games:", item_meta["item_key"].str.startswith("game:rawg_").sum())
display(item_meta[["item_key", "title", "domain", "description"]].head())

f:\Questro-ASP.NET-REACT\machine-learning\cl_epidtn_recommender_improved_8.py:498: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  adult_tags = tags["tag"].astype(str).str.contains(r'\b(NSFW|Nudity|Sexual Content|Adult|sex)\b', case=False, na=False)
f:\Questro-ASP.NET-REACT\machine-learning\cl_epidtn_recommender_improved_8.py:520: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  adult_tags = meta["tags"].astype(str).str.contains(r'\b(NSFW|Nudity|Sexual Content|Hentai|Adult|sex)\b', case=False, na=False)
f:\Questro-ASP.NET-REACT\machine-learning\cl_epidtn_recommender_improved_8.py:293: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  adult_themes = adult_themes.str.contains(r'\b(NSFW|Nudity|Sexual Content|Adult|sex)\b', case=False, 

interactions: (2924246, 4)
item_meta: (138457, 25)
HF enrichment stats: {'movie_rows': 87585, 'movie_matches': 55124, 'game_rows': 50872, 'game_matches': 10126}

Catalog breakdown:
domain
movie    87585
game     50872
Name: count, dtype: int64

TMDB-only movies: 0
RAWG-only games: 0


,item_key,title,domain,description
0,movie:1,Toy Story (1995),movie,"Led by Woody, Andy's toys live happily in his ..."
1,movie:2,Jumanji (1995),movie,When siblings Judy and Peter discover an encha...
2,movie:3,Grumpier Old Men (1995),movie,A family wedding reignites the ancient feud be...
3,movie:4,Waiting to Exhale (1995),movie,"Cheated on, mistreated and stepped on, the wom..."
4,movie:5,Father of the Bride Part II (1995),movie,Just when George Banks has recovered from his ...


## Survey Data

`users_ratings.csv` is parsed into explicit profile ratings and can be used for leave-one-out evaluation or as positive fine-tuning interactions.

In [5]:
survey_profiles_unmapped, genre_vocab = load_survey_user_profiles(cfg.survey_csv)
survey_positive_interactions = build_survey_interactions(cfg.survey_csv)
print("survey profiles:", len(survey_profiles_unmapped))
print("genre vocab:", len(genre_vocab))
print("positive / intent interactions:", survey_positive_interactions.shape)
display(survey_positive_interactions.head())

survey profiles: 106
genre vocab: 32
positive / intent interactions: (507, 6)


,user_key,item_key,timestamp,domain,rating_value,rating_weight
0,survey:0,movie:132046,0,movie,3.5,0.25
1,survey:0,game:723390,1,game,3.5,0.25
2,survey:0,game:271590,2,game,5.0,1.00
3,survey:0,game:1871750,3,game,3.5,0.25
4,survey:0,game:502500,5,game,3.5,0.25


## Build Temporal Splits

The improved split uses older events for training, the penultimate event for validation, and the final event for test.

In [6]:
# Optional: include survey positive/intent events in the training universe.
combined_interactions = pd.concat(
    [interactions, survey_positive_interactions[["user_key", "item_key", "timestamp", "domain"]]],
    ignore_index=True,
).drop_duplicates()

seq = make_temporal_sequence_splits(combined_interactions, cfg, all_item_keys=item_meta["item_key"])
survey_profiles, genre_vocab = load_survey_user_profiles(cfg.survey_csv, seq["item_to_idx"])
print("survey catalog coverage:", survey_catalog_coverage(survey_profiles))
print("train samples:", len(seq["histories"]))
print("validation rows:", len(seq["validation_rows"]))
print("test rows:", len(seq["test_rows"]))
print("users:", len(seq["user_to_idx"]))
print("items:", max(seq["item_to_idx"].values()))

survey catalog coverage: {'ratings': 1060, 'mapped_ratings': 1060, 'unique_items': 805, 'mapped_unique_items': 805, 'coverage': 1.0}
train samples: 1000000
validation rows: 25836
test rows: 25836
users: 25836
items: 138457


## Metadata Tensors and Optional Text Embeddings

The HF-enriched text is already in `item_meta['description']` and `item_meta['tokens']`. Sentence-transformer embeddings are optional and require `sentence-transformers` to be installed.

In [7]:
token_to_idx, item_token_ids, item_domain_ids, title_lookup = encode_item_metadata(item_meta, seq["item_to_idx"])

try:
    item_text_embeddings = build_text_embedding_tensor(
        item_meta,
        seq["item_to_idx"],
        cache_path=ARTIFACTS / "improved_item_text_embeddings.pt",
    )
except RuntimeError as exc:
    print("Skipping sentence-transformer embeddings:", exc)
    item_text_embeddings = None

train_ds = SequenceDataset(seq["histories"], seq["targets"], seq["users"], seq["activity"], cfg.max_seq_len)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)

print("metadata tokens:", len(token_to_idx))
print("text embeddings:", None if item_text_embeddings is None else tuple(item_text_embeddings.shape))

Batches: 100%|██████████| 1082/1082 [01:30<00:00, 11.91it/s]


metadata tokens: 59897
text embeddings: (138458, 384)


## Train Improved CL-EPIDTN

The improved training loop adds AMM-style PAV mimic loss and gradient clipping. Training for 10 epochs.

In [8]:
model = CLEPIDTN(
    n_items=max(seq['item_to_idx'].values()) + 1,
    n_users=len(seq['user_to_idx']),
    n_tokens=len(token_to_idx),
    item_token_ids=item_token_ids,
    item_domain_ids=item_domain_ids,
    item_text_embeddings=item_text_embeddings,
    cfg=cfg,
).to(cfg.device)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=1e-4)
scheduler = build_training_scheduler(optimizer, train_loader, cfg)

for epoch in range(cfg.epochs):
    loss = train_one_epoch(model, train_loader, optimizer, cfg,
                           scheduler=scheduler,
                           desc=f'epoch {epoch + 1}/{cfg.epochs}')
    item_index = build_item_index(model)
    val_metrics = evaluate_retrieval(model, item_index, seq['validation_rows'], cfg, ks=(10, 50), max_users=1000)
    print(f'epoch={epoch + 1} loss={loss:.4f}', val_metrics)


epoch=1 loss=5.7065 {'users': 1000.0, 'MRR': 0.017696970870418775, 'HR@10': 0.049, 'Recall@10': 0.049, 'NDCG@10': 0.022520872381080713, 'HR@50': 0.13, 'Recall@50': 0.13, 'NDCG@50': 0.03944452929666321}


epoch=2 loss=5.2025 {'users': 1000.0, 'MRR': 0.026444863827311472, 'HR@10': 0.053, 'Recall@10': 0.053, 'NDCG@10': 0.029358084046016492, 'HR@50': 0.146, 'Recall@50': 0.146, 'NDCG@50': 0.049750996001791725}


epoch=3 loss=5.0501 {'users': 1000.0, 'MRR': 0.022259299903063745, 'HR@10': 0.057, 'Recall@10': 0.057, 'NDCG@10': 0.025762557301379278, 'HR@50': 0.179, 'Recall@50': 0.179, 'NDCG@50': 0.05259180774580283}


epoch=4 loss=4.9502 {'users': 1000.0, 'MRR': 0.026272448406624762, 'HR@10': 0.069, 'Recall@10': 0.069, 'NDCG@10': 0.03269690417754457, 'HR@50': 0.176, 'Recall@50': 0.176, 'NDCG@50': 0.05550103836958611}


epoch=5 loss=4.8707 {'users': 1000.0, 'MRR': 0.024095662648662346, 'HR@10': 0.057, 'Recall@10': 0.057, 'NDCG@10': 0.027892783347931018, 'HR@50': 0.166, 'Recall@50': 0.166, 'NDCG@50': 0.051522794129239595}


epoch=6 loss=4.8005 {'users': 1000.0, 'MRR': 0.029162110708110155, 'HR@10': 0.076, 'Recall@10': 0.076, 'NDCG@10': 0.03579561887840434, 'HR@50': 0.198, 'Recall@50': 0.198, 'NDCG@50': 0.06184587592411641}


epoch=7 loss=4.7389 {'users': 1000.0, 'MRR': 0.025800950100428374, 'HR@10': 0.065, 'Recall@10': 0.065, 'NDCG@10': 0.03059755951012317, 'HR@50': 0.182, 'Recall@50': 0.182, 'NDCG@50': 0.05619070436037895}


epoch=8 loss=4.6854 {'users': 1000.0, 'MRR': 0.02722681293739785, 'HR@10': 0.071, 'Recall@10': 0.071, 'NDCG@10': 0.033237599666619155, 'HR@50': 0.182, 'Recall@50': 0.182, 'NDCG@50': 0.0575975940818192}


epoch=9 loss=4.6448 {'users': 1000.0, 'MRR': 0.029726559329353353, 'HR@10': 0.072, 'Recall@10': 0.072, 'NDCG@10': 0.03554329601677144, 'HR@50': 0.183, 'Recall@50': 0.183, 'NDCG@50': 0.059636877848131375}


epoch=10 loss=4.6237 {'users': 1000.0, 'MRR': 0.0298543807846327, 'HR@10': 0.071, 'Recall@10': 0.071, 'NDCG@10': 0.03533673276309249, 'HR@50': 0.185, 'Recall@50': 0.185, 'NDCG@50': 0.06007105856608437}


## Baselines

Compare the trained model with simple popularity and token-overlap Item-KNN baselines.

In [9]:
domain_by_key = dict(zip(item_meta["item_key"], item_meta["domain"]))
pop_rankings = build_popularity_rankings(combined_interactions, seq["item_to_idx"], domain_by_key)
print(evaluate_popularity_baseline(pop_rankings, seq["validation_rows"], ks=(10, 50)))

token_knn = build_token_knn_index(item_token_ids)
example_user, example_history, example_target = seq["validation_rows"][0]
print("target:", title_lookup.get(example_target, example_target))
for item_id, score in recommend_item_knn(example_history, token_knn, k=10):
    print(f"{score:.3f}", title_lookup.get(item_id, item_id))

{'users': 25836.0, 'Popularity_HR@10': 0.0823269856014863, 'Popularity_NDCG@10': 0.036329723852651076, 'Popularity_HR@50': 0.2899055581359344, 'Popularity_NDCG@50': 0.08048977163494965}


target: Citizen Ruth (1996)


11.004 Godfather, The (1972)
9.438 Fargo (1996)
9.369 Green Mile, The (1999)
9.346 Schindler's List (1993)
9.208 Slumdog Millionaire (2008)
9.185 Out of Sight (1998)
9.185 Paper Moon (1973)
9.116 Untouchables, The (1987)
9.093 Mystic River (2003)
9.070 Gone with the Wind (1939)


## Survey Leave-One-Out Evaluation

This hides each survey user's last positive/intended item and recommends from the earlier positives.

In [10]:
item_index = build_item_index(model)
print(evaluate_survey_leave_one_out(survey_profiles, model, item_index, cfg, k=10))

{'Survey_HR@10': 0.0, 'survey_users': 96.0}


## Survey User-Based Collaborative Filtering

This path uses `users_ratings.csv` directly. It finds survey users with similar explicit ratings and liked/disliked genre profiles, then recommends unseen items that those neighbors rated positively. Because the survey has only 106 users and sparse item overlap, treat this as a complementary signal and baseline rather than a replacement for the neural model.

In [11]:
user_cf_metrics = evaluate_user_cf_leave_one_out(
    survey_profiles,
    k=10,
    neighbor_count=15,
)
print(user_cf_metrics)

target_profile = survey_profiles[0]
user_cf_recommendations, similar_users = recommend_user_cf(
    target_profile,
    survey_profiles,
    k=10,
    neighbor_count=15,
    domain=None,  # use "movie" or "game" to filter
)

print("nearest survey users:")
for neighbor in similar_users[:5]:
    print(
        neighbor["user_key"],
        f"similarity={neighbor['similarity']:.3f}",
        f"shared_items={int(neighbor['shared_items'])}",
    )

print("\nuser-CF recommendations:")
for item_id, score in user_cf_recommendations:
    print(f"{score:.3f}", title_lookup.get(item_id, item_id))

print("\nhybrid neural + user-CF recommendations:")
hybrid_recommendations, hybrid_details = recommend_hybrid_survey_user(
    target_profile,
    survey_profiles,
    model,
    item_index,
    cfg,
    k=10,
    domain=None,
    text_index=item_text_embeddings,
    neural_weight=0.75,
    user_cf_weight=0.25,
)
for item_id, score in hybrid_recommendations:
    print(f"{score:.4f}", title_lookup.get(item_id, item_id))


{'UserCF_HR@10': 0.010416666666666666, 'UserCF_MRR@10': 0.001736111111111111, 'user_cf_users': 96.0, 'user_cf_coverage': 1.0}
nearest survey users:
survey:67 similarity=0.611 shared_items=0
survey:49 similarity=0.589 shared_items=0
survey:73 similarity=0.471 shared_items=0
survey:42 similarity=0.458 shared_items=0
survey:103 similarity=0.441 shared_items=0

user-CF recommendations:
0.667 Up (2009)
0.500 Toy Story 3 (2010)
0.500 Madagascar (2005)
0.500 Horizon Zero Dawn™ Complete Edition
0.500 Toy Story (1995)
0.500 Hyper Light Drifter
0.500 Enforcer: Police Crime Action
0.500 Night at the Museum (2006)
0.500 The Hobbit: The Battle of the Five Armies (2014)
0.500 Cavern of Dreams

hybrid neural + user-CF recommendations:
0.0357 Borderlands Game of the Year
0.0341 Guns and Robots
0.0326 Grand Theft Auto III – The Definitive Edition
0.0312 PlayZ
0.0300 Mirror's Edge™ Catalyst
0.0288 Haydee 2
0.0278 Journey
0.0268 Just Cause™ 3
0.0259 Just Cause
0.0250 Borderlands: The Pre-Sequel


## Anonymous Recommendation

In [12]:
rated_titles = {
    "Batman Begins": 5,
    "Batman, The": 5,
    # "Batman & Robin": 1,
}
history, history_weights = anonymous_history_from_ratings(rated_titles, item_meta, seq["item_to_idx"])
recommendations = recommend_from_history(
    model,
    item_index,
    history,
    user_id=None,
    activity=len(history),
    cfg=cfg,
    k=10,
    domain="movie",
    text_index=item_text_embeddings,
    text_weight=0.45 if item_text_embeddings is not None else 0.0,
    history_weights=history_weights,
)
for item_id, score in recommendations:
    print(f"{score:.3f}", title_lookup.get(item_id, item_id))

2.163 Dark Knight, The (2008)
1.948 Dark Knight Rises, The (2012)
1.873 Sin City (2005)
1.827 Batman: Mask of the Phantasm (1993)
1.769 Batman: The Long Halloween, Part Two (2021)
1.746 The Batman (2022)
1.699 Incredibles, The (2004)
1.675 Batman (1989)
1.654 Batman vs. Robin (2015)
1.599 The Death of Superman


## Cross-Domain Movie to Game Recommendation

For movie-to-game cold start, the content-first cross-domain ranker is still useful because public MovieLens and public Steam users are not shared identities.

In [13]:
game_recommendations = recommend_cross_domain_content(
    item_meta,
    seq["item_to_idx"],
    history_ids=history,
    target_domain="game",
    k=10,
    text_index=item_text_embeddings,
    min_reviews=500,
    exclude_addons=True,
    history_weights=history_weights,
)
for item_id, score in game_recommendations:
    print(f"{score:.3f}", title_lookup.get(item_id, item_id))

1.528 Batman - The Telltale Series
1.447 Batman: Arkham City - Game of the Year Edition
1.422 Batman™: Arkham Knight
1.412 Gotham Knights
1.344 Batman: The Enemy Within - The Telltale Series
1.293 Batman™: Arkham Origins
1.263 Batman: Arkham Asylum Game of the Year Edition
1.219 LEGO® Batman™ 2: DC Super Heroes
1.187 LEGO® Batman™ 3: Beyond Gotham
1.150 Batman™: Arkham Origins Blackgate - Deluxe Edition


In [14]:
print('zebi')

zebi


## Use TMDB and RAWG IDs

TMDB IDs map directly through MovieLens `links.csv`. RAWG IDs are available after HF enrichment and are matched to the Steam catalog by normalized title, because this RAWG cache does not store Steam app IDs. You can also pass native MovieLens IDs with `movie`/`movielens` and Steam app IDs with `game`/`steam`.

In [16]:
# Example external-ID ratings: (source, external_id, rating_on_1_to_5_scale)
# TMDB 155 = The Dark Knight, RAWG 3498 = Grand Theft Auto V in many RAWG dumps.
external_ratings = [
    ("tmdb", 155, 5),
    ("rawg", 3498, 5),
    ("steam", 271590, 5),      # native Steam app id
    ("movielens", 58559, 5),   # native MovieLens movie id
]

history_ids, rating_values, resolved_items = history_from_external_ratings(
    external_ratings,
    item_meta,
    seq["item_to_idx"],
    ignore_missing=True,
)
print("resolved inputs:")
for item in resolved_items:
    print(item)

recommendations, resolved_items = recommend_from_external_ratings(
    model,
    item_index,
    external_ratings,
    item_meta,
    seq["item_to_idx"],
    cfg,
    k=10,
    domain=None,       # use "movie" or "game" to force one domain
    text_index=item_text_embeddings,
    ignore_missing=True,
)
print("\nrecommendations:")
for item_id, score in recommendations:
    print(f"{score:.3f}", title_lookup.get(item_id, item_id))

resolved inputs:
{'item_id': 128458, 'item_key': 'movie:58559', 'title': 'Dark Knight, The (2008)', 'domain': 'movie', 'source': 'tmdb', 'external_id': 155}
{'item_id': 27582, 'item_key': 'game:271590', 'title': 'Grand Theft Auto V', 'domain': 'game', 'source': 'rawg', 'external_id': 3498}
{'item_id': 27582, 'item_key': 'game:271590', 'title': 'Grand Theft Auto V', 'domain': 'game', 'source': 'steam', 'external_id': 271590}
{'item_id': 128458, 'item_key': 'movie:58559', 'title': 'Dark Knight, The (2008)', 'domain': 'movie', 'source': 'movielens', 'external_id': 58559}

recommendations:
1.964 Batman Begins (2005)
1.885 Inception (2010)
1.875 Batman: Arkham City - Game of the Year Edition
1.865 Saints Row: The Third
1.821 Wolfenstein: The New Order
1.806 Gotham Knights
1.803 Just Cause™ 3
1.775 Alan Wake's American Nightmare
1.772 E.Y.E: Divine Cybermancy
1.760 BlazBlue: Cross Tag Battle


## Save Checkpoint

In [17]:
# ============================================================
# Save all artifacts needed by recommender_api_improved8.py
# ============================================================
ARTIFACTS.mkdir(exist_ok=True)

# 1. Build the final item index (shape: n_items x d, PAD at index 0)
item_index = build_item_index(model)

# 2. Save the full model object (recommender_api.py expects checkpoint['model'])
torch.save(
    {
        'model': model,  # full object for API loading
        'model_state': model.state_dict(),
        'cfg': cfg,
        'item_to_idx': seq['item_to_idx'],
        'user_to_idx': seq['user_to_idx'],
        'token_to_idx': token_to_idx,
        'title_lookup': title_lookup,
        'item_text_embeddings': item_text_embeddings,
        'hf_enrichment_stats': item_meta.attrs.get('hf_enrichment_stats'),
        'survey_coverage': survey_catalog_coverage(survey_profiles),
    },
    ARTIFACTS / 'improved_5epochs.pt',
)
print('Saved improved_5epochs.pt')

# 3. Save item_index tensor
torch.save(item_index, ARTIFACTS / 'item_index.pt')
print('Saved item_index.pt  shape:', tuple(item_index.shape))

# 4. Save item_meta DataFrame
with open(ARTIFACTS / 'item_meta.pkl', 'wb') as _f:
    pickle.dump(item_meta, _f)
print('Saved item_meta.pkl  rows:', len(item_meta))

# 5. Save item_to_idx dict
with open(ARTIFACTS / 'item_to_idx.pkl', 'wb') as _f:
    pickle.dump(seq['item_to_idx'], _f)
print('Saved item_to_idx.pkl  entries:', len(seq['item_to_idx']))

# 6. Save title_lookup dict
with open(ARTIFACTS / 'title_lookup.pkl', 'wb') as _f:
    pickle.dump(title_lookup, _f)
print('Saved title_lookup.pkl  entries:', len(title_lookup))

print('\nAll API artifacts saved to:', ARTIFACTS)
print('Point recommender_api_improved8.py CONFIG at this directory.')


Saved improved_5epochs.pt
Saved item_index.pt  shape: (138458, 128)
Saved item_meta.pkl  rows: 138457
Saved item_to_idx.pkl  entries: 138458
Saved title_lookup.pkl  entries: 138457

All API artifacts saved to: f:\Questro-ASP.NET-REACT\machine-learning\data\artifacts_improved8
Point recommender_api_improved8.py CONFIG at this directory.


## Bake In the Recommender Catalog (cold-add, no retrain)

The RAG service (`recommender/`) retrieves from a FAISS store whose items are the
output of `recommender/src/pipeline` filtering (`unify_and_format_domain` +
`deduplicate`, persisted to `vector_store/metadata.db`). Many of those items were
never seen at training time, so the API used to *hot-add* them per request — which
grows the model's GPU tables while scoring runs and trips a CUDA device-side assert
("indices beyond safe range"), capping how many can be added.

This cell does that registration **once, offline, on a clean context**, and writes
the result back to the artifacts, so the API never hot-adds. New items are
cold-start: zero learned `item_id` embedding, but a real tokenized row (using the
existing `token_to_idx` vocab), a MiniLM text embedding, and an `item_index` row
produced by the item tower — exactly how the trained rows were built.

It is self-contained (loads the saved artifacts from disk), so you can run it
without re-running training. Source of truth is `metadata.db`, guaranteeing the ML
catalog is filtered identically to what the RAG can retrieve.

In [ ]:
# === COLD-ADD: bake the recommender catalog into the artifacts ===
import json, sqlite3, gc, pickle
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from cl_epidtn_recommender_improved_8 import (
    PAD, _movie_genre_bridge_tokens, _text_tokens, _title_tokens,
)

ART      = Path("artifacts_improved8")                    # where the saved artifacts live
META_DB  = Path("../recommender/vector_store/metadata.db")  # recommender's filtered catalog
CKPT     = ART / "improved_8epochs.pt"
ENCODER  = "sentence-transformers/all-MiniLM-L6-v2"        # must match the trained text dim (384)
MAX_TOKENS, NARR_LIMIT = 80, 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- load artifacts; free the checkpoint's redundant copies early (RAM) ---
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
model = ck["model"]; token_to_idx = ck["token_to_idx"]; model.eval()
ck.pop("model_state", None); ck.pop("item_text_embeddings", None)
model.item_text_embeddings = None  # duplicate of the text file loaded below
gc.collect()

item_to_idx  = pickle.load(open(ART / "item_to_idx.pkl", "rb"))
title_lookup = pickle.load(open(ART / "title_lookup.pkl", "rb"))
item_meta    = pickle.load(open(ART / "item_meta.pkl", "rb"))
item_index   = torch.load(ART / "item_index.pt", map_location="cpu", weights_only=False)
_te = torch.load(ART / "improved_item_text_embeddings.pt", map_location="cpu", weights_only=False)
text_index = _te["tensor"] if isinstance(_te, dict) else _te
print("current catalog:", f"{len(item_to_idx):,}", "items")

def metadata_key(d):
    rid = str(d.get("id", ""))
    if "_" not in rid: return None
    _, actual = rid.split("_", 1); actual = actual.strip()
    if not actual: return None
    return ("movie:" if d.get("type") == "movie" else "game:") + actual

rows = sqlite3.connect(str(META_DB)).execute("SELECT data FROM metadata").fetchall()
new = {}
for (raw,) in rows:
    try: d = json.loads(raw)
    except Exception: continue
    k = metadata_key(d)
    if k and k not in item_to_idx and k not in new:
        new[k] = d
print("new items to add:", f"{len(new):,}")

def build_token_string(domain, themes, narrative, title):
    parts = []
    t = _text_tokens(themes)
    if t: parts.append(t)
    if domain == "movie":
        b = _movie_genre_bridge_tokens(str(themes).replace(",", "|"))
        if b.strip(): parts.append(b)
    n = _text_tokens(narrative, limit=NARR_LIMIT)
    if n: parts.append(n)
    ti = _title_tokens(title)        # title last: title_boost reads the tail
    if ti: parts.append(ti)
    return " ".join(parts)

if new:
    new_keys = sorted(new)
    n_old = max(item_to_idx.values()) + 1
    n_new = len(new_keys); n_total = n_old + n_new
    cols = list(item_meta.columns)
    tok_new = np.zeros((n_new, MAX_TOKENS), dtype=np.int64)
    dom_new = np.zeros(n_new, dtype=np.int64)
    texts, meta_rows = [], []
    for i, k in enumerate(new_keys):
        d = new[k]; idx = n_old + i
        domain = "movie" if d.get("type") == "movie" else "game"
        _, actual = d["id"].split("_", 1)
        title = d.get("title") or k
        themes = d.get("themes", "") or ""; narrative = d.get("narrative", "") or ""
        ts = build_token_string(domain, themes, narrative, title)
        toks = [token_to_idx.get(t, 1) for t in ts.split()[:MAX_TOKENS]]
        tok_new[i, :len(toks)] = toks
        dom_new[i] = 0 if domain == "movie" else 1
        item_to_idx[k] = idx; title_lookup[idx] = title
        texts.append(f"{title}. {narrative} {themes}".strip())
        row = {c: pd.NA for c in cols}
        row.update(item_key=k, title=title, domain=domain, tokens=ts, description=narrative,
                   hf_description=narrative, hf_genres=themes, hf_tags="",
                   is_adult=bool(d.get("is_adult", False)),
                   tmdb_id=int(actual) if domain == "movie" and actual.isdigit() else pd.NA,
                   rawg_id=int(actual) if domain == "game" and actual.isdigit() else pd.NA)
        meta_rows.append(row)

    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer(ENCODER, device=str(device)).encode(
        texts, batch_size=64, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True)
    assert enc.shape[1] == text_index.shape[1], (enc.shape, text_index.shape)
    text_full = torch.cat([text_index, torch.tensor(enc, dtype=text_index.dtype)], 0)
    del enc; gc.collect()

    model.item_token_ids = torch.cat(
        [model.item_token_ids.cpu(), torch.tensor(tok_new, dtype=model.item_token_ids.dtype)], 0)
    model.item_domain_ids = torch.cat(
        [model.item_domain_ids.cpu(), torch.tensor(dom_new, dtype=model.item_domain_ids.dtype)], 0)
    _old = model.item_id
    _new = nn.Embedding(n_total, _old.embedding_dim, padding_idx=PAD)
    with torch.no_grad():
        _new.weight.zero_(); _new.weight[:_old.num_embeddings].copy_(_old.weight.data.cpu())
    model.item_id = _new
    model.item_text_embeddings = text_full

    model.to(device)
    rows_new = []
    with torch.no_grad():
        for s in range(n_old, n_total, 4096):
            ids = torch.arange(s, min(s + 4096, n_total), device=device)
            rows_new.append(model.item_features(ids).cpu())
    item_index_full = torch.cat([item_index] + rows_new, 0)
    model.to("cpu")
    model.item_token_ids = model.item_token_ids.cpu()
    model.item_domain_ids = model.item_domain_ids.cpu()
    model.item_text_embeddings = model.item_text_embeddings.cpu()

    item_meta_full = pd.concat([item_meta, pd.DataFrame(meta_rows, columns=cols)],
                               ignore_index=True, sort=False)
    assert item_index_full.shape[0] == n_total == text_full.shape[0] == model.item_id.num_embeddings
    print("catalog grown:", f"{n_old:,}", "->", f"{n_total:,}")

    pickle.dump(item_to_idx,  open(ART / "item_to_idx.pkl", "wb"))
    pickle.dump(title_lookup, open(ART / "title_lookup.pkl", "wb"))
    pickle.dump(item_meta_full, open(ART / "item_meta.pkl", "wb"))
    torch.save(item_index_full, ART / "item_index.pt")
    torch.save({"tensor": text_full}, ART / "improved_item_text_embeddings.pt")
    ck["model"] = model; ck["model_state"] = model.state_dict()
    ck["item_to_idx"] = item_to_idx; ck["title_lookup"] = title_lookup
    ck["item_text_embeddings"] = text_full
    torch.save(ck, CKPT)
    print("Saved augmented artifacts to", ART, "| total items:", f"{n_total:,}")
else:
    print("Nothing to add - every catalog item is already in the model.")


In [18]:
import zipfile

zip_path = ARTIFACTS / "artifacts_improved8.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(ARTIFACTS.glob("*")):
        if f.name.endswith(".zip"):
            continue  # don't zip the zip
        print(f"  adding {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")
        zf.write(f, arcname=f.name)

print(f"\n✅ {zip_path.name}  ({zip_path.stat().st_size / 1e6:.1f} MB)")

# Auto-download in Colab
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Not in Colab — zip saved to:", zip_path)


  adding improved_5epochs.pt  (433.1 MB)
  adding improved_item_text_embeddings.pt  (316.3 MB)
  adding item_index.pt  (70.9 MB)
  adding item_meta.pkl  (168.8 MB)
  adding item_to_idx.pkl  (2.6 MB)
  adding title_lookup.pkl  (4.3 MB)

✅ artifacts_improved8.zip  (678.5 MB)
Not in Colab — zip saved to: f:\Questro-ASP.NET-REACT\machine-learning\data\artifacts_improved8\artifacts_improved8.zip
